# JevDecider Demo

Loads a Qwen-based JEV decision model from Hugging Face and demonstrates:

1. raw `choice`, `noul` and `score` calls
2. the JEV-style `decide` call (one shared state, several typed questions)
3. the batched forms

In [ ]:
# %pip install -e ..
import json
import time

from jev_berta import JevDecider

MODEL_ID = "leobitz/decision-master-base-label-encode-0.1"
decider = JevDecider.from_pretrained(MODEL_ID, batch_size=32)  # tune batch_size to your GPU memory
print("architecture:", decider.architecture, "| device:", decider.device)


def show(obj):
    print(json.dumps(obj, indent=2))

ImportError: cannot import name 'JevDecider' from 'jev_berta' (/home/leo/Downloads/variable_decision_data_builder_actions/jev-berta/src/jev_berta/__init__.py)

## Choice

Options can be a plain list or a `{key: description}` mapping. Include a catch-all such as `needs_review`.

In [ ]:
context = "My card was charged twice for the same invoice last week."
query = "Which team should handle this request?"
choices = {
    "billing": "Payments, invoices, refunds, or payouts",
    "technical": "Bugs, outages, or integration failures",
    "sales": "Pricing, upgrades, or new accounts",
    "needs_review": "The request does not clearly fit another option",
}

show(decider.choice(context, query, choices))

## Noul (yes/no)

`noul` is P(yes). The question is asked with every yes/no wording seen in training (yes/no, true/false, correct/incorrect, ...) in both orders, and the results are averaged.

In [ ]:
refund_query = "Does this message explicitly request a refund?"

show(decider.noul(context, refund_query))
show(decider.noul("What is your pricing for 50 seats?", refund_query))

## Score

Levels are ordered low to high. The score is probability-weighted, so it can fall between levels (numeric keys are used as values, otherwise 0-based indices).

In [ ]:
levels = {
    "routine": "No time pressure; can wait for normal handling",
    "elevated": "Blocks a user or needs same-day attention",
    "critical": "Outage or data loss affecting many users",
}

show(decider.score("The whole site has been down for all customers since 9am.", "How urgent is this request?", levels))

## JEV-style `decide`

One shared state, several independent questions. The state can be text, a JSON object, or a list. Answers come back under `answers`, keyed by question id.

In [ ]:
state = {
    "ticket": "My card was charged twice for the same invoice last week.",
    "account_tier": "enterprise",
}
questions = {
    "department": {"type": "choice", "instructions": "Which team should handle this request?", "criteria": choices},
    "urgency": {"type": "score", "instructions": "How urgent is this request?", "criteria": levels},
    "refund": {"type": "noul", "instructions": refund_query},
}

result = decider.decide(state, questions)
show(result)

### Policy stays in application code

Thresholds below are placeholders; choose real ones from a labeled evaluation set.

In [ ]:
ALLOWED = set(choices) - {"needs_review"}
dept = result["answers"]["department"]

if dept["choice"] in ALLOWED and dept["confidence"] >= 0.7:
    print("route to:", dept["choice"])
else:
    print("send to human review")

if result["answers"]["refund"]["noul"] >= 0.5:
    print("flag: refund requested (still check authorization before acting)")

## Batched forms

Items are dicts. All variants of all items are packed into shared model batches; answers come back in input order.

In [ ]:
tickets = [
    "My card was charged twice for the same invoice last week.",
    "The API returns a 500 error whenever I upload a file.",
    "Can I upgrade from the starter plan to the team plan?",
]

routed = decider.choice_batch([{"context": t, "query": query, "choices": choices} for t in tickets])
refunds = decider.noul_batch([{"context": t, "query": refund_query} for t in tickets])

for t, r, n in zip(tickets, routed, refunds):
    print(f"{r['choice']:<13} conf={r['confidence']:.2f}  refund={n['noul']:.2f}  | {t}")

In [ ]:
# decide_batch: a different state per request, each with its own questions.
answers = decider.decide_batch([{"state": t, "questions": questions} for t in tickets])

for t, a in zip(tickets, answers):
    ans = a["answers"]
    print(f"{ans['department']['choice']:<13} urgency={ans['urgency']['score']:.2f}  refund={ans['refund']['noul']:.2f}  | {t}")

### Looping vs batching

In [ ]:
items = [{"context": t, "query": refund_query} for t in tickets * 4]

start = time.perf_counter()
looped = [decider.noul(i["context"], i["query"]) for i in items]
t_loop = time.perf_counter() - start

start = time.perf_counter()
batched = decider.noul_batch(items)
t_batch = time.perf_counter() - start

print(f"loop: {t_loop:.2f}s | batch: {t_batch:.2f}s | {len(items)} items")
print("max |difference|:", max(abs(a["noul"] - b["noul"]) for a, b in zip(looped, batched)))

## Loading any checkpoint

`jev_berta.load` picks `JevDecider` or `JevBerta` from the checkpoint's `config.json`.

In [ ]:
from jev_berta import load

model = load(MODEL_ID, batch_size=32)
print(type(model).__name__)